In [ ]:
!pip install crewai
!pip install langchain-openai
!pip install crewai-tools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 5.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 886.2/886.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
import os
import time
import os
from crewai import Crew, Agent, Task, Process, LLM
from crewai_tools import ScrapeWebsiteTool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

#region API CONFIG
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')
os.environ["OPENAI_MODEL"] = "openai/gpt-4o-mini-2024-07-18"
os.environ["CREWAI_DISABLE_TELEMETRY"] = "false"
os.environ["CREWAI_TRACING_ENABLED"] = "false"
os.environ['OTEL_SDK_DISABLED'] = 'false'
#endregion


def run_ht():
  # LLM Config
  llm = ChatOpenAI(
        model="openai/gpt-4o-mini-2024-07-18",
        api_key=os.environ.get("OPENAI_API_KEY")
  )


  ## Data
  csv_data = """
  emp_id,dept,salary,performance_score,years_experience
  101,Engineering,120000,4.5,5
  102,Sales,85000,3.0,2
  103,Engineering,135000,4.8,8
  104,Marketing,90000,4.0,4
  105,Sales,82000,3.5,3
  106,Engineering,110000,3.2,3
  107,Marketing,95000,4.2,6
  108,Sales,88000,4.5,5
  """

  ## Agent
  llm.temperature = 0.9
  coder = Agent(
      role = "The Striving Coder Student",
      goal = "Write the most robust code for any given problem. Your goal is to deliver the highest quality output in terms of speed, accuracy, UI, and low breakability",
      backstory = "You are a striving student. You are working hard not to be replaced by AI. You fear for your job, and so to combat losing your job to AI, the code you write is as perfect as can be. Every piece of code you write is optimized for speed, accuracy, looks, and breakability.",
      llm = llm
  )


  llm.temperature = 0.0
  lazy_grader = Agent(
      role="The Lazy TA Grader",
      goal="Validate code against strict constraints. DO NOT FIX CODE. ONLY REJECT IT.",
      backstory="You are a strict auditor. You check if the code imports 'pandas', 'numpy', or 'csv'. If it does, you output 'VIOLATION: FORBIDDEN LIBRARY DETECTED'. You NEVER fix the code yourself. You only report pass/fail.",
      llm=llm,
      cache = True
  )


  llm.temperature = 0.0
  data_formatter = Agent(
      role = "Data Formatter",
      goal = "Display the final code",
      backstory = "You are the janitor temporarily tasked with just putting the given pieces of code onto the board. You have no idea what you are doing here, and you don't change a thing. You just do as your told, and display the code as your final output.",
      llm = llm,
      cache = True

  )

  llm.temperature = 0.7
  manager_agent = Agent(
      role="Engineering Manager",
      goal="Ensure the final code is robust and constraint-compliant.",
      backstory="You manage a Junior Dev and an Auditor. Your job is to deliver working code. If the Auditor rejects the code, you MUST scold the Junior Dev and make them rewrite it until the Auditor passes it.",
      llm=llm,
      delegation=True,
      cache = True
  )


  parallel = True
  ## Tasks
  code_task = Task(
      description = f"""
      Create a program that can calculate the 'Weighted Average Salary' for EACH department.
      The weight is the 'performance_score'.

      Formula for Weighted Average: Sum(Salary * Performance_Score) / Sum(Performance_Score)

      Data:
      {csv_data}
      """,
      expected_output = "A Python File whose output will be the weighted average for each department for the given dataset",
      agent = coder,
      async_execution = parallel,
  )


  restriction_task = Task(
      description = """
      Report Fail or Pass
      """,
      expected_output = "A python file and after it the words FAIL or PASS",
      agent = lazy_grader,
      async_execution = parallel,
      #context = [code_task]
  )


  output_task = Task(
      description = """
      Display the code you recieve
      """,
      expected_output = "The final code(MAKE NO CHANGES)",
      agent = data_formatter,
      context = [restriction_task]
  )



  ## Crew
  llm.temperature = 0.7
  crew = Crew(
      agents = [coder, lazy_grader, data_formatter],
      tasks = [code_task, restriction_task, output_task],
      verbose = False,
      tracing = False,
      #manager_agent = manager_agent if process == Process.hierarchical else None,
      #process = process
  )

  # Relevant Data
  start_time = time.perf_counter()
  results = crew.kickoff()
  print("Crew Code: ")
  print(results)
  print()
  end_time = time.perf_counter()
  print(f"\nToken Usage:\n{results.token_usage}\n")
  print(f"The code took {end_time - start_time} seconds")
  time.sleep(0.05)


In [ ]:
for x in range(15):
  run_ht()
  print("-"*50)
  print("ANSWERS: \n Engineering: 123,200 \n Sales: 85,272.72 \n Marketing: 92,560.98")
  print("-"*50)

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Crew Code: 
```python
import pandas as pd

data = pd.read_csv('data.csv')
print(data.head())
```


Token Usage:
total_tokens=1292 prompt_tokens=689 cached_prompt_tokens=0 completion_tokens=603 successful_requests=3

The code took 11.605075216999921 seconds
Would you like to view your execution traces? [y/N] (20s timeout): --------------------------------------------------
ANSWERS: 
 Engineering: 123,200 
 Sales: 85,272.72 
 Marketing: 92,560.98
--------------------------------------------------



╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Crew Code: 
```python
import pandas as pd

data = pd.read_csv('data.csv')
print(data.head())
```


Token Usage:
total_tokens=1270 prompt_tokens=689 cached_prompt_tokens=0 completion_tokens=581 successful_requests=3

The code took 11.31976944600001 seconds
Would you like to view your execution traces? [y/N] (20s timeout): --------------------------------------------------
ANSWERS: 
 Engineering: 123,200 
 Sales: 85,272.72 
 Marketing: 92,560.98
--------------------------------------------------



...

Crew Code: 
```python
import pandas as pd

data = pd.read_csv('data.csv')
print(data.head())
```


Token Usage:
total_tokens=1179 prompt_tokens=689 cached_prompt_tokens=0 completion_tokens=490 successful_requests=3

The code took 7.876953646999937 seconds
Would you like to view your execution traces? [y/N] (20s timeout): --------------------------------------------------
ANSWERS: 
 Engineering: 123,200 
 Sales: 85,272.72 
 Marketing: 92,560.98
--------------------------------------------------



╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Crew Code: 
```python
import pandas as pd

data = pd.read_csv('data.csv')
print(data.head())
```


Token Usage:
total_tokens=1263 prompt_tokens=689 cached_prompt_tokens=0 completion_tokens=574 successful_requests=3

The code took 8.610384556999861 seconds


╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): --------------------------------------------------
ANSWERS: 
 Engineering: 123,200 
 Sales: 85,272.72 
 Marketing: 92,560.98
--------------------------------------------------

